# NB11 - Federated Learning for LLMs
## Background
Federated Instruction Tuning (FedIT, Zhang et al., 2023) applies federated learning to fine-tune large language models. Communication is the key bottleneck: transmitting full model gradients for a 7B model requires ~28GB per round. LoRA (Low-Rank Adaptation) reduces this: instead of transmitting the full gradient matrix, clients only share low-rank matrices A and B (rank r << d), cutting communication by d/r.

FedLoRA aggregates LoRA adapters across clients, then merges them into the base model using weighted averaging. This enables fine-tuning billion-parameter models across organizations without sharing raw data or full model weights.

## What You'll Learn
- LoRA parameterization: W' = W + alpha * B @ A, where A, B are low-rank
- Communication reduction analysis: full gradients vs LoRA
- FedLoRA: federated aggregation of LoRA adapters
- Non-IID effects in federated LLM fine-tuning

## Why This Matters
FedLoRA enables privacy-preserving fine-tuning of LLMs across healthcare, finance, and legal sectors where data cannot leave institutional boundaries. The LoRA communication compression makes federated LLM fine-tuning practically feasible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
from dataclasses import dataclass, field

# ── LoRA: Low-Rank Adaptation ─────────────────────────────────────────────
@dataclass
class LoRAAdapter:
    rank: int
    alpha: float
    A: np.ndarray  # (rank, d_in)
    B: np.ndarray  # (d_out, rank)

    @classmethod
    def initialize(cls, d_in: int, d_out: int, rank: int,
                   alpha: float = 1.0, seed: int = 42):
        np.random.seed(seed)
        A = np.random.randn(rank, d_in) * 0.02
        B = np.zeros((d_out, rank))  # B init to 0 -> adapter starts at 0
        return cls(rank, alpha, A, B)

    def get_delta_W(self) -> np.ndarray:
        """Delta weight matrix = alpha/rank * B @ A."""
        return (self.alpha / self.rank) * self.B @ self.A

    def n_params(self) -> int:
        return self.A.size + self.B.size

def communication_analysis(d_in: int, d_out: int, ranks: List[int]):
    full_params = d_in * d_out
    print(f'Full weight matrix: {d_in}x{d_out} = {full_params:,} params')
    print()
    print(f'{'Rank':>6} | {'LoRA params':>12} | {'Reduction':>10} | {'Comm MB (fp16)':>14}')
    print('-' * 50)
    for r in ranks:
        lora_params = (r * d_in) + (d_out * r)
        reduction = full_params / lora_params
        comm_mb = lora_params * 2 / 1e6  # fp16 = 2 bytes
        print(f'{r:>6} | {lora_params:>12,} | {reduction:>10.1f}x | {comm_mb:>14.2f}')

communication_analysis(4096, 4096, ranks=[4, 8, 16, 32, 64])

In [ ]:
# ── FedLoRA simulation ─────────────────────────────────────────────────────
@dataclass
class FedLoRAClient:
    client_id: int
    X: np.ndarray
    y: np.ndarray
    W_base: np.ndarray
    lora: LoRAAdapter

    def local_train(self, n_steps: int = 20, lr: float = 0.01) -> float:
        """Train LoRA adapter on local data."""
        n = len(self.X)
        total_loss = 0
        for step in range(n_steps):
            # Forward: W_effective = W_base + delta_W
            W_eff = self.W_base + self.lora.get_delta_W()
            logits = self.X @ W_eff.T  # (n, n_classes)
            logits -= logits.max(axis=1, keepdims=True)
            probs = np.exp(logits); probs /= probs.sum(axis=1, keepdims=True)
            loss = -np.mean(np.log(probs[np.arange(n), self.y] + 1e-10))
            total_loss += loss
            # Gradient
            dlogits = probs.copy(); dlogits[np.arange(n), self.y] -= 1; dlogits /= n
            dW_eff = dlogits.T @ self.X  # (n_classes, d_in)
            # LoRA gradient
            scale = self.lora.alpha / self.lora.rank
            dB = scale * dW_eff @ self.lora.A.T
            dA = scale * self.lora.B.T @ dW_eff
            self.lora.B -= lr * dB
            self.lora.A -= lr * dA
        return total_loss / n_steps

def fedlora_aggregate(client_adapters: List[LoRAAdapter],
                       weights: List[float] = None) -> LoRAAdapter:
    """Federated averaging of LoRA adapters."""
    if weights is None:
        weights = [1.0 / len(client_adapters)] * len(client_adapters)
    agg_A = sum(w * a.A for w, a in zip(weights, client_adapters))
    agg_B = sum(w * a.B for w, a in zip(weights, client_adapters))
    return LoRAAdapter(client_adapters[0].rank, client_adapters[0].alpha, agg_A, agg_B)

# ── Setup: heterogeneous clients ───────────────────────────────────────────
np.random.seed(42)
d_in, n_classes, rank = 8, 3, 4
W_base = np.random.randn(n_classes, d_in) * 0.3

# 4 clients with different data distributions (non-IID)
n_clients = 4
clients = []
for cid in range(n_clients):
    # Each client has data skewed to different classes
    n_c = 50
    dominant_class = cid % n_classes
    y_c = np.array([dominant_class] * (n_c//2) +
                    [(dominant_class + 1) % n_classes] * (n_c//4) +
                    [(dominant_class + 2) % n_classes] * (n_c//4))
    X_c = np.zeros((n_c, d_in))
    for i, yi in enumerate(y_c):
        X_c[i] = np.eye(n_classes, d_in)[yi] * 2 + np.random.randn(d_in) * 0.5
    lora = LoRAAdapter.initialize(d_in, n_classes, rank, alpha=1.0, seed=cid)
    clients.append(FedLoRAClient(cid, X_c, y_c, W_base.copy(), lora))

# ── Global test set ────────────────────────────────────────────────────────
X_test = np.vstack([np.eye(n_classes, d_in)[c] * 2 + np.random.randn(50, d_in) * 0.5
                    for c in range(n_classes)])
y_test = np.repeat(np.arange(n_classes), 50)

def eval_model(W_base, lora, X_te, y_te):
    W_eff = W_base + lora.get_delta_W()
    return (np.argmax(X_te @ W_eff.T, axis=1) == y_te).mean()

# ── Federated rounds ───────────────────────────────────────────────────────
n_rounds = 10
global_lora = LoRAAdapter.initialize(d_in, n_classes, rank, alpha=1.0, seed=99)
round_accs = []

for r in range(n_rounds):
    # Distribute global adapter to clients
    for client in clients:
        client.lora.A = global_lora.A.copy()
        client.lora.B = global_lora.B.copy()
    # Local training
    for client in clients:
        client.local_train(n_steps=10, lr=0.05)
    # Aggregate
    n_samples = [len(c.X) for c in clients]
    total = sum(n_samples)
    weights = [n/total for n in n_samples]
    global_lora = fedlora_aggregate([c.lora for c in clients], weights)
    acc = eval_model(W_base, global_lora, X_test, y_test)
    round_accs.append(acc)
    print(f'Round {r+1:2d}: accuracy = {acc:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, n_rounds+1), round_accs, '-o', linewidth=2)
ax.set_title('FedLoRA: Global Accuracy per Round')
ax.set_xlabel('Federated round'); ax.set_ylabel('Test accuracy')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE}/s16_11_fedlora.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Communication per round (per client): {global_lora.n_params()*2/1024:.1f} KB')
print(f'vs Full gradient: {d_in*n_classes*4/1024:.1f} KB (fp32)')
print(f'Reduction: {d_in*n_classes*4 / (global_lora.n_params()*2):.1f}x')